In [21]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False


In [22]:
DATA_PATH = "..\\Data\\01-02_원료_전처리와_제선_제선조업.csv"


In [23]:
# 1. csv에서 datetime 데이터 불러오기: to_datetime() 이용
df = pd.read_csv(DATA_PATH, encoding="utf-8")

print(df.shape)
print("timestamp의 데이터타입(1):", df["timestamp"].dtype)

df["timestamp"] = pd.to_datetime(df["timestamp"])
print("timestamp의 데이터타입(2):", df["timestamp"].dtype)
print()
print(df.head())


(720, 6)
timestamp의 데이터타입(1): str
timestamp의 데이터타입(2): datetime64[us]

            timestamp  blast_flow_nm3min  blast_pressure_kpa  \
0 2026-03-02 06:00:00             5213.5               381.2   
1 2026-03-02 06:01:00             5155.1               383.0   
2 2026-03-02 06:02:00             5189.5               379.0   
3 2026-03-02 06:03:00             5229.3               377.5   
4 2026-03-02 06:04:00             5201.2               379.2   

   top_pressure_kpa  hot_blast_temp_c  blower_vib_mms  
0             214.0            1182.0            3.46  
1             210.2            1184.3            3.36  
2             209.2            1184.8            3.43  
3             207.3            1179.1            3.44  
4             214.7            1177.9            3.55  


In [24]:
# 2. read_csv()의 parse_dates 옵션 이용
df = pd.read_csv(DATA_PATH, encoding="utf-8", parse_dates=["timestamp"])

print("timestamp의 데이터타입(3):", df["timestamp"].dtype)
print(df.shape)


timestamp의 데이터타입(3): datetime64[us]
(720, 6)


In [25]:
# timestamp의 시간 간격 확인
gaps = df["timestamp"].diff().value_counts()
print(gaps)

# 720행의 데이터 중 서로 인접한 719개의 시간 간격이 전부 1분이다.


timestamp
0 days 00:01:00    719
Name: count, dtype: int64


In [26]:
# 송풍량, 송풍압, 송풍기 진동의 기초 통계
cols = ["blast_flow_nm3min", "blast_pressure_kpa", "blower_vib_mms"]
print(df[cols].describe().round(1))


       blast_flow_nm3min  blast_pressure_kpa  blower_vib_mms
count              720.0               720.0           720.0
mean              5088.2               388.8             3.4
std                159.6                13.0             0.1
min               4681.8               372.8             3.2
25%               4977.5               379.4             3.3
50%               5180.8               381.7             3.4
75%               5202.5               398.3             3.4
max               5258.2               421.4             3.6


In [27]:
# 이동 평균: 송풍량의 장기적인 방향 확인
df["flow_ma"] = df["blast_flow_nm3min"].rolling(window=15).mean()

print(df["flow_ma"].head(3).tolist())
print(round(df["flow_ma"].iloc[14], 1), round(df["flow_ma"].iloc[400], 1))


[nan, nan, nan]
5201.5 5200.8


In [28]:
# 이동 표준편차: 노정압 흔들림 확인
df["top_sd"] = df["top_pressure_kpa"].rolling(window=30).std()

print(round(df["top_sd"].iloc[200], 2), round(df["top_sd"].iloc[560], 2))


2.64 4.28


In [29]:
# 문제 14. 앞 6시간과 뒤 6시간 평균 비교
before = df.iloc[:360]
after = df.iloc[360:]

compare = pd.DataFrame({
    "앞_6시간_평균": before[cols].mean(),
    "뒤_6시간_평균": after[cols].mean(),
})
compare["변화량"] = compare["뒤_6시간_평균"] - compare["앞_6시간_평균"]


def direction(diff):
    if abs(diff) < 0.01:
        return "거의 변화 없음"
    if diff > 0:
        return "증가"
    return "감소"


compare["변화_방향"] = compare["변화량"].map(direction)
compare.index = ["송풍량", "송풍압", "송풍기 진동"]

print(compare.round(3).to_string())


        앞_6시간_평균  뒤_6시간_평균      변화량     변화_방향
송풍량     5198.668  4977.828 -220.840        감소
송풍압      379.787   397.716   17.929        증가
송풍기 진동     3.397     3.399    0.001  거의 변화 없음


In [30]:
# 제출용 요약
answer = pd.DataFrame({
    "항목": ["송풍량", "송풍압", "송풍기 진동"],
    "변화 방향": [
        compare.loc["송풍량", "변화_방향"],
        compare.loc["송풍압", "변화_방향"],
        compare.loc["송풍기 진동", "변화_방향"],
    ],
    "해석": [
        "뒤 6시간 평균이 낮아졌다.",
        "뒤 6시간 평균이 높아졌다.",
        "평균 차이가 매우 작아 설비 진동 변화는 크지 않다.",
    ],
})

answer


,항목,변화 방향,해석
0,송풍량,감소,뒤 6시간 평균이 낮아졌다.
1,송풍압,증가,뒤 6시간 평균이 높아졌다.
2,송풍기 진동,거의 변화 없음,평균 차이가 매우 작아 설비 진동 변화는 크지 않다.
